# Practical 7: Understanding Regression with T-Test and ANOVA

### Teachers' Rating Dataset

This practical uses the **TeachingRatings** dataset to study:
1. Whether gender affects teaching evaluation scores using linear regression and a t-test.
2. Whether beauty scores differ across age groups using one-way ANOVA.
3. Whether teaching evaluation score is related to beauty score.

> **Dataset:** TeachingRatings (463 observations).  
> **Practical questions and required outputs:** based on the supplied Practical 7 PDF. fileciteturn0file0L2-L10


### Dataset
This practical uses the supplied `TeachingRatings.csv` file (463 observations and 13 columns). Keep the CSV in the same folder as this notebook before running it.

## 1. Import Required Libraries

We use:
- **Pandas** for data handling.
- **Statsmodels** for OLS regression and ANOVA.
- **SciPy** for the Pearson correlation test.
- **Matplotlib / Seaborn** for a simple visualization.


In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:.6f}")


## 2. Load the Teachers' Rating Dataset

The dataset contains 463 course observations and includes variables such as `age`, `gender`, `beauty`, and `eval`.

If your teacher has provided a local CSV file, replace the URL with the file name, for example:

```python
df = pd.read_csv("TeachingRatings.csv")
```


In [2]:
# Load the uploaded TeachingRatings dataset
import pandas as pd

df = pd.read_csv('TeachingRatings.csv')
print('Dataset shape:', df.shape)
df.head()


Dataset shape: (463, 13)


,rownames,minority,age,gender,credits,beauty,eval,division,native,tenure,students,allstudents,prof
0,1,yes,36,female,more,0.289916,4.300000,upper,yes,yes,24,43,1
1,2,no,59,male,more,-0.737732,4.500000,upper,yes,yes,17,20,2
2,3,no,51,male,more,-0.571984,3.700000,upper,yes,yes,55,55,3
3,4,no,40,female,more,-0.677963,4.300000,upper,yes,yes,40,46,4
4,5,no,31,female,more,1.509794,4.400000,upper,yes,yes,42,48,5


**Short explanation:**  
The dataset is successfully loaded. The important variables for this practical are `eval` (teaching evaluation), `gender`, `beauty`, and `age`.


# Q1. Regression with T-Test

### Question
**Using the teachers' rating data set, does gender affect teaching evaluation rates?** fileciteturn0file0L3-L5

### Hypotheses
- **H₀:** Gender does not affect the mean teaching evaluation score.
- **H₁:** Gender affects the mean teaching evaluation score.

We fit the model:

\[
eval = \beta_0 + \beta_1(female) + \epsilon
\]

The t-test for the gender coefficient checks whether `β₁` is significantly different from zero.


In [3]:
# Create a binary gender variable: female = 1, male = 0
df["female"] = (df["gender"] == "female").astype(int)

# OLS regression: teaching evaluation on gender
model_gender = smf.ols("eval ~ female", data=df).fit()

# Display the regression output
print(model_gender.summary())


                            OLS Regression Results                            
Dep. Variable:                   eval   R-squared:                       0.022
Model:                            OLS   Adj. R-squared:                  0.020
Method:                 Least Squares   F-statistic:                     10.56
Date:                Wed, 11 Nov 2026   Prob (F-statistic):            0.00124
Time:                        17:19:35   Log-Likelihood:                -378.50
No. Observations:                 463   AIC:                             761.0
Df Residuals:                     461   BIC:                             769.3
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      4.0690      0.034    121.288      0.0

### Short interpretation

The regression output gives a **t-statistic and p-value** for `female`.

- If the p-value is below 0.05, the gender coefficient is statistically significant.
- In the required output, the `female` coefficient is approximately **-0.168**, with **t ≈ -3.25** and **p ≈ 0.001**.
- Therefore, in this sample, the estimated mean evaluation score for female instructors is about **0.168 points lower than for male instructors** in this simple regression.

This is an association in the dataset; it does not by itself establish that gender causes the difference.


# Q2. Regression with ANOVA

### Question
**Using the teachers' rating data set, does beauty score for instructors differ by age?** fileciteturn0file0L6-L10

ANOVA requires categorical groups, so age is divided into three groups as used in the practical material:

- **40 years and younger:** age ≤ 40
- **between 40 and 57 years:** 40 < age < 57
- **57 years and older:** age ≥ 57

### Hypotheses
- **H₀:** The mean beauty score is the same for all three age groups.
- **H₁:** At least one age-group mean is different.


In [4]:
# Create the three age groups

df["age_group"] = np.select(
    [
        df["age"] <= 40,
        (df["age"] > 40) & (df["age"] < 57),
        df["age"] >= 57
    ],
    [
        "Young",
        "Middle",
        "Older"
    ],
    default="Unknown"
)

df[["age", "age_group"]].head()


,age,age_group
0,36,Young
1,59,Older
2,51,Middle
3,40,Young
4,31,Young


In [5]:
# Regression model for ANOVA
model_age = smf.ols("beauty ~ C(age_group)", data=df).fit()

# One-way ANOVA table
anova_results = anova_lm(model_age, typ=2)
anova_results


,sum_sq,df,F,PR(>F)
C(age_group),20.422743,2.000000,17.597558,0.000000
Residual,266.925153,460.000000,NaN,NaN


### Required output from the practical sheet

| Term | df | sum_sq | mean_sq | F | PR(>F) |
|---|---:|---:|---:|---:|---:|
| age_group | 2.0 | 20.422744 | 10.211372 | 17.597559 | 4.322549e-08 |

The practical sheet specifies this ANOVA output. fileciteturn0file0L9-L10

### Short interpretation

The p-value is extremely small (**4.322549 × 10⁻⁸**), which is below 0.05. Therefore, the null hypothesis is rejected: **the mean beauty score is not the same across all three age groups**. ANOVA tells us that at least one group differs; it does not by itself identify which specific pairs differ.


# Q3. Correlation / Regression

### Question
**Using the teachers' rating dataset, is teaching evaluation score correlated with beauty score?** fileciteturn0file0L13-L15

The supplied practical output uses an OLS regression of `eval` on `beauty`:

\[
eval = \beta_0 + \beta_1(beauty) + \epsilon
\]

We also calculate Pearson's correlation coefficient to directly measure the linear correlation.


In [6]:
# OLS regression: evaluation score on beauty score
model_beauty = smf.ols("eval ~ beauty", data=df).fit()

print(model_beauty.summary())


                            OLS Regression Results                            
Dep. Variable:                   eval   R-squared:                       0.036
Model:                            OLS   Adj. R-squared:                  0.034
Method:                 Least Squares   F-statistic:                     17.08
Date:                Wed, 11 Nov 2026   Prob (F-statistic):           4.25e-05
Time:                        17:19:35   Log-Likelihood:                -375.32
No. Observations:                 463   AIC:                             754.6
Df Residuals:                     461   BIC:                             762.9
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      3.9983      0.025    157.727      0.0

In [7]:
# Pearson correlation
r, p_value = pearsonr(df["beauty"], df["eval"])

print(f"Pearson correlation (r): {r:.6f}")
print(f"p-value: {p_value:.6g}")


Pearson correlation (r): 0.189039
p-value: 4.24712e-05


### Short interpretation

The required regression output shows:

- **Beauty coefficient ≈ 0.1330**
- **t ≈ 4.133**
- **p ≈ 4.25 × 10⁻⁵**
- **R² ≈ 0.036**

Because the p-value is below 0.05, there is statistically significant evidence of a **positive linear association** between beauty score and teaching evaluation in this sample. The R² is small, so beauty alone explains only a small proportion of the variation in evaluation scores. fileciteturn0file0L13-L16


# Conclusion

In this practical:

1. **Gender and teaching evaluation:** The simple regression gives a statistically significant gender coefficient (`p ≈ 0.001`), with the female coefficient estimated at about `-0.168`.
2. **Beauty and age group:** One-way ANOVA gives `F ≈ 17.598` and `p ≈ 4.32 × 10⁻⁸`, indicating that mean beauty scores differ across the defined age groups.
3. **Beauty and teaching evaluation:** The regression/correlation analysis shows a statistically significant positive association, but the small `R² ≈ 0.036` indicates that beauty alone explains only a limited amount of variation in teaching evaluations.

**Overall:** The statistical tests identify associations in the teachers' rating dataset. These results should not be interpreted as proof of causation.
